# GeoMarketing IDF — J5 : équipements et services

**Objectif :** enrichir le fichier du J4 avec les équipements et services de la Base permanente des équipements (BPE) 2025.

Ce notebook :

1. télécharge le fichier communal officiel de l'Insee ;
2. conserve les communes d'Île-de-France en géographie 2026 ;
3. calcule les grandes familles d'équipements ;
4. crée des indicateurs utiles à un projet d'implantation en restauration rapide ;
5. joint les résultats au J4 sans supprimer les anciennes colonnes ;
6. exporte une table complète pour le J5.

Source : [Insee — Dénombrement des équipements BPE 2025](https://www.insee.fr/fr/statistiques/8217527?sommaire=8217537), publié le 9 juillet 2026.

> Les restaurants de la BPE réunissent la restauration traditionnelle et la restauration rapide. Cette variable est donc un indicateur général d'activité, pas encore une mesure précise de la concurrence.

## 1. Préparer Python

Si une ancienne importation a échoué, utilise **Kernel → Restart Kernel**, puis relance toutes les cellules.

In [ ]:
import os

for variable in ("OMP_NUM_THREADS", "NUMEXPR_NUM_THREADS", "MKL_NUM_THREADS"):
    valeur = os.environ.get(variable)
    if valeur is not None:
        try:
            int(valeur)
        except ValueError:
            print(f"Variable invalide supprimée : {variable}={valeur!r}")
            os.environ.pop(variable, None)

from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import ZipFile
import io
import shutil

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 70)
pd.set_option("display.max_rows", 100)
print("Environnement Python prêt.")

## 2. Retrouver le projet et le fichier du J4

Modifie uniquement `DOSSIER_PROJET_MANUEL` si la détection automatique échoue.

In [ ]:
DOSSIER_PROJET_MANUEL = None
# Exemple : Path(r"C:\Users\VotreNom\OneDrive\GeoMarketing_IDF")

def trouver_projet():
    if DOSSIER_PROJET_MANUEL is not None:
        return Path(DOSSIER_PROJET_MANUEL)

    candidats = []
    for variable in ("OneDrive", "OneDriveConsumer", "OneDriveCommercial"):
        racine = os.environ.get(variable)
        if racine:
            candidats.extend([
                Path(racine) / "GeoMarketing_IDF",
                Path(racine) / "Documents" / "GeoMarketing_IDF",
            ])

    candidats.extend([
        Path.home() / "OneDrive" / "GeoMarketing_IDF",
        Path.home() / "Documents" / "GeoMarketing_IDF",
        Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd(),
    ])

    for candidat in candidats:
        if (candidat / "data").exists():
            return candidat.resolve()

    raise FileNotFoundError(
        "Dossier GeoMarketing_IDF introuvable. Renseigne DOSSIER_PROJET_MANUEL."
    )

PROJET = trouver_projet()
RAW = PROJET / "data" / "raw" / "insee"
PROCESSED = PROJET / "data" / "processed" / "insee"
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

FICHIER_J4 = PROCESSED / "profil_geomarketing_idf_j4.csv"
ARCHIVE_BPE = RAW / "DS_BPE_CSV_FR.zip"
FICHIER_SORTIE = PROCESSED / "profil_geomarketing_idf_j5.csv"

print("Projet :", PROJET)
print("Entrée J4 :", FICHIER_J4)
print("Archive BPE :", ARCHIVE_BPE)
print("Sortie J5 :", FICHIER_SORTIE)
assert FICHIER_J4.exists(), f'Fichier J4 absent : {FICHIER_J4}'

## 3. Télécharger et ouvrir la BPE 2025

Le fichier compressé fait environ 13 Mo. Il reste compressé afin d'éviter de stocker inutilement environ 150 Mo dans OneDrive.

In [ ]:
URL_BPE = "https://www.insee.fr/fr/statistiques/fichier/8217527/DS_BPE_CSV_FR.zip"
FORCER_TELECHARGEMENT = False

if FORCER_TELECHARGEMENT or not ARCHIVE_BPE.exists():
    print("Téléchargement de la BPE 2025...")
    fichier_temporaire = ARCHIVE_BPE.with_suffix(".tmp")
    requete = Request(URL_BPE, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urlopen(requete, timeout=240) as reponse, open(fichier_temporaire, "wb") as sortie:
            shutil.copyfileobj(reponse, sortie)
        assert fichier_temporaire.stat().st_size > 5_000_000, "Téléchargement incomplet."
        with ZipFile(fichier_temporaire) as z:
            mauvais = z.testzip()
            assert mauvais is None, f"Fichier corrompu dans l'archive : {mauvais}"
        fichier_temporaire.replace(ARCHIVE_BPE)
    finally:
        if fichier_temporaire.exists():
            fichier_temporaire.unlink()
    print("Téléchargement terminé.")
else:
    print("Archive déjà présente :", ARCHIVE_BPE)

def detecter_fichiers_bpe(archive):
    requis_donnees = {"GEO", "GEO_OBJECT", "FACILITY_DOM", "FACILITY_SDOM", "FACILITY_TYPE", "OBS_VALUE"}
    requis_metadata = {"COD_VAR", "COD_MOD", "LIB_MOD"}
    donnees = None
    metadata = None
    with ZipFile(archive) as z:
        for info in z.infolist():
            if not info.filename.lower().endswith(".csv"):
                continue
            texte = z.read(info.filename)[:10000].decode("utf-8-sig", errors="replace")
            sep = ";" if texte.splitlines()[0].count(";") > texte.splitlines()[0].count(",") else ","
            colonnes = set(pd.read_csv(io.StringIO(texte), sep=sep, nrows=0).columns.str.strip())
            if requis_donnees.issubset(colonnes):
                donnees = (info.filename, sep)
            elif requis_metadata.issubset(colonnes):
                metadata = (info.filename, sep)
    assert donnees is not None, "CSV de données BPE non reconnu."
    assert metadata is not None, "CSV de métadonnées BPE non reconnu."
    return donnees, metadata

(nom_donnees, sep_donnees), (nom_metadata, sep_metadata) = detecter_fichiers_bpe(ARCHIVE_BPE)
print("Données :", nom_donnees)
print("Métadonnées :", nom_metadata)

## 4. Charger uniquement les communes franciliennes

Le fichier national est lu par morceaux afin de limiter la mémoire utilisée.

In [ ]:
idf_departements = {"75", "77", "78", "91", "92", "93", "94", "95"}

with ZipFile(ARCHIVE_BPE) as z:
    morceaux = []
    with z.open(nom_donnees) as fichier:
        for bloc in pd.read_csv(
            fichier, sep=sep_donnees, dtype=str, encoding="utf-8-sig",
            chunksize=250_000, low_memory=False
        ):
            masque = (
                bloc["GEO_OBJECT"].eq("COM")
                & bloc["GEO"].str[:2].isin(idf_departements)
                & bloc["TIME_PERIOD"].eq("2025")
            )
            morceaux.append(bloc.loc[masque].copy())
    bpe = pd.concat(morceaux, ignore_index=True)

    with z.open(nom_metadata) as fichier:
        metadata = pd.read_csv(
            fichier, sep=sep_metadata, dtype=str, encoding="utf-8-sig", low_memory=False
        )

bpe.columns = bpe.columns.str.strip()
metadata.columns = metadata.columns.str.strip()
bpe["CODGEO"] = bpe["GEO"].astype("string").str.strip().str.zfill(5)
bpe["OBS_VALUE_NUM"] = pd.to_numeric(
    bpe["OBS_VALUE"].str.replace(" ", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce",
)

lib_communes = (
    metadata.loc[
        metadata["COD_VAR"].eq("GEO") & metadata["GEO_OBJECT"].eq("COM"),
        ["COD_MOD", "LIB_MOD"],
    ]
    .drop_duplicates("COD_MOD")
    .rename(columns={"COD_MOD": "CODGEO", "LIB_MOD": "NOM_COMMUNE_BPE"})
)
bpe = bpe.merge(lib_communes, on="CODGEO", how="left", validate="many_to_one")

print(f"{len(bpe):,} lignes BPE conservées.")
print("Communes BPE franciliennes :", bpe["CODGEO"].nunique())
assert bpe["CODGEO"].str.fullmatch(r"\d{5}").all()
assert "75056" in set(bpe["CODGEO"]), "Paris commune (75056) est absent."
assert "93066" in set(bpe["CODGEO"]), "Saint-Denis (93066) est absent."
assert "93059" not in set(bpe["CODGEO"]), "Pierrefitte apparaît encore séparément."
display(bpe.head())

## 5. Calculer les sept grandes familles d'équipements

On utilise les lignes de total déjà fournies par l'Insee, ce qui évite de compter deux fois les sous-totaux et les types détaillés.

In [ ]:
noms_domaines = {
    "A": "NB_SERVICES_PARTICULIERS",
    "B": "NB_COMMERCES",
    "C": "NB_ENSEIGNEMENT",
    "D": "NB_SANTE_ACTION_SOCIALE",
    "E": "NB_TRANSPORTS_BPE",
    "F": "NB_SPORT_LOISIRS_CULTURE",
    "G": "NB_TOURISME",
}

total = (
    bpe.loc[
        bpe["FACILITY_DOM"].eq("_T")
        & bpe["FACILITY_SDOM"].eq("_T")
        & bpe["FACILITY_TYPE"].eq("_T"),
        ["CODGEO", "NOM_COMMUNE_BPE", "OBS_VALUE_NUM"],
    ]
    .drop_duplicates("CODGEO")
    .rename(columns={"OBS_VALUE_NUM": "NB_EQUIPEMENTS_TOTAL"})
)

domaines = bpe.loc[
    bpe["FACILITY_DOM"].isin(noms_domaines)
    & bpe["FACILITY_SDOM"].eq("_T")
    & bpe["FACILITY_TYPE"].eq("_T"),
    ["CODGEO", "FACILITY_DOM", "OBS_VALUE_NUM"],
].copy()

domaines = domaines.pivot_table(
    index="CODGEO", columns="FACILITY_DOM", values="OBS_VALUE_NUM", aggfunc="first"
).rename(columns=noms_domaines).reset_index()

equipements = total.merge(domaines, on="CODGEO", how="outer", validate="one_to_one")
print("Communes agrégées :", len(equipements))
display(equipements.head())

## 6. Créer les indicateurs stratégiques

Les codes utilisés sont ceux de la nomenclature BPE 2025. Les regroupements sont affichés pour rester vérifiables.

In [ ]:
groupes_types = {
    "NB_ETABLISSEMENTS_SCOLAIRES": ["C107", "C108", "C109", "C201", "C301", "C302", "C303"],
    "NB_ENSEIGNEMENT_SUPERIEUR": ["C401", "C403", "C409", "C410", "C411", "C501", "C502", "C503", "C504", "C505", "C509"],
    "NB_ETABLISSEMENTS_HOSPITALIERS": ["D101", "D102", "D103", "D104", "D105", "D106", "D107"],
    "NB_MEDECINS_GENERALISTES": ["D265"],
    "NB_PHARMACIES": ["D307"],
    "NB_CINEMAS": ["F303"],
    "NB_HOTELS": ["G102"],
    "NB_SUPERMARCHES_HYPERMARCHES": ["B104", "B105"],
    "NB_BOULANGERIES_PATISSERIES": ["B207"],
    "NB_RESTAURANTS_RESTAURATION_RAPIDE_BPE": ["A504"],
}

groupes_sous_domaines = {
    "NB_COMMERCES_ALIMENTAIRES": ["B2"],
    "NB_EQUIPEMENTS_SPORTIFS": ["F1"],
    "NB_EQUIPEMENTS_CULTURELS": ["F3"],
}

types_disponibles = set(bpe["FACILITY_TYPE"])
for nom, codes in groupes_types.items():
    absents = set(codes) - types_disponibles
    assert not absents, f"Codes BPE absents pour {nom} : {sorted(absents)}"
    extrait = (
        bpe.loc[bpe["FACILITY_TYPE"].isin(codes)]
        .groupby("CODGEO", as_index=False)["OBS_VALUE_NUM"].sum()
        .rename(columns={"OBS_VALUE_NUM": nom})
    )
    equipements = equipements.merge(extrait, on="CODGEO", how="left", validate="one_to_one")

for nom, codes in groupes_sous_domaines.items():
    extrait = (
        bpe.loc[
            bpe["FACILITY_SDOM"].isin(codes) & bpe["FACILITY_TYPE"].eq("_T")
        ]
        .groupby("CODGEO", as_index=False)["OBS_VALUE_NUM"].sum()
        .rename(columns={"OBS_VALUE_NUM": nom})
    )
    equipements = equipements.merge(extrait, on="CODGEO", how="left", validate="one_to_one")

colonnes_equipements = [c for c in equipements.columns if c.startswith("NB_")]
equipements[colonnes_equipements] = equipements[colonnes_equipements].fillna(0).round().astype("int64")

assert equipements["CODGEO"].is_unique
display(equipements.head())

## 7. Contrôler les codes et libellés stratégiques

In [ ]:
codes_strategiques = sorted({code for codes in groupes_types.values() for code in codes})
libelles_types = metadata.loc[
    metadata["COD_VAR"].eq("FACILITY_TYPE") & metadata["COD_MOD"].isin(codes_strategiques),
    ["COD_MOD", "LIB_MOD"],
].drop_duplicates().sort_values("COD_MOD")
display(libelles_types)
print("Nombre de codes stratégiques contrôlés :", len(libelles_types))

## 8. Joindre les équipements au fichier complet du J4

La jointure part du J4 : toutes les anciennes colonnes sont conservées.

In [ ]:
def lire_csv_robuste(chemin):
    for encodage in ("utf-8-sig", "utf-8", "cp1252"):
        for sep in (";", ","):
            try:
                df = pd.read_csv(chemin, sep=sep, dtype=str, encoding=encodage)
                if len(df.columns) > 1:
                    return df
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue
    raise ValueError(f"Impossible de lire {chemin}")

profil_j4 = lire_csv_robuste(FICHIER_J4)
profil_j4.columns = profil_j4.columns.str.strip()
assert "CODGEO" in profil_j4.columns, "CODGEO absent du fichier J4."
profil_j4["CODGEO"] = profil_j4["CODGEO"].astype("string").str.strip().str.zfill(5)
assert profil_j4["CODGEO"].is_unique

codes_bpe_hors_j4 = sorted(set(equipements["CODGEO"]) - set(profil_j4["CODGEO"]))
print("Codes BPE absents du J4 :", len(codes_bpe_hors_j4))
if codes_bpe_hors_j4:
    print(codes_bpe_hors_j4[:20])

profil_j5 = profil_j4.merge(
    equipements,
    on="CODGEO",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("Communes du J4 :", len(profil_j4))
print("Communes du J5 :", len(profil_j5))
print("Colonnes du J4 :", len(profil_j4.columns))
print("Colonnes du J5 :", len(profil_j5.columns) - 1)
assert len(profil_j5) == len(profil_j4), "Le nombre de communes a changé."

non_jointes = profil_j5["_merge"].ne("both")
print("Communes sans correspondance BPE :", non_jointes.sum())
if non_jointes.any():
    display(profil_j5.loc[non_jointes, ["CODGEO", "_merge"]].head(20))
profil_j5 = profil_j5.drop(columns="_merge")

for col in colonnes_equipements:
    profil_j5[col] = pd.to_numeric(profil_j5[col], errors="coerce").fillna(0).astype("int64")
profil_j5["NOM_COMMUNE_BPE"] = profil_j5["NOM_COMMUNE_BPE"].fillna("")

## 9. Calculer les densités par habitant

Les nombres bruts mesurent le volume. Les ratios permettent de comparer des communes de tailles différentes.

In [ ]:
def trouver_population(df):
    for col in ("POP_2022", "P22_POP", "POPULATION_2022"):
        if col in df.columns:
            return col
    return None

col_population = trouver_population(profil_j5)
assert col_population is not None, "Colonne de population introuvable."
profil_j5[col_population] = pd.to_numeric(
    profil_j5[col_population].astype("string").str.replace(" ", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce",
)

ratios = {
    "EQUIPEMENTS_POUR_10000_HAB": "NB_EQUIPEMENTS_TOTAL",
    "COMMERCES_POUR_10000_HAB": "NB_COMMERCES",
    "ENSEIGNEMENT_POUR_10000_HAB": "NB_ENSEIGNEMENT",
    "SANTE_SOCIAL_POUR_10000_HAB": "NB_SANTE_ACTION_SOCIALE",
    "SPORT_LOISIRS_CULTURE_POUR_10000_HAB": "NB_SPORT_LOISIRS_CULTURE",
    "RESTAURANTS_BPE_POUR_10000_HAB": "NB_RESTAURANTS_RESTAURATION_RAPIDE_BPE",
}

for nouveau_nom, colonne_nombre in ratios.items():
    profil_j5[nouveau_nom] = (
        10_000 * profil_j5[colonne_nombre] / profil_j5[col_population]
    ).round(2)

display(profil_j5[list(ratios)].describe().T)

## 10. Contrôler quelques communes

In [ ]:
assert "75056" in set(profil_j5["CODGEO"]), "Paris (75056) est absent."
assert "93066" in set(profil_j5["CODGEO"]), "Saint-Denis (93066) est absent."
assert "93059" not in set(profil_j5["CODGEO"]), "Pierrefitte apparaît encore séparément."

communes_test = ["75056", "93066", "95018", "95680", "95268"]
colonnes_affichage = [
    c for c in [
        "CODGEO", "LIBGEO", "NOM_COM", "NOM_COMMUNE_BPE",
        "NB_EQUIPEMENTS_TOTAL", "NB_COMMERCES", "NB_ENSEIGNEMENT",
        "NB_SANTE_ACTION_SOCIALE", "NB_SPORT_LOISIRS_CULTURE",
        "NB_ETABLISSEMENTS_SCOLAIRES", "NB_ENSEIGNEMENT_SUPERIEUR",
        "NB_CINEMAS", "NB_HOTELS", "NB_SUPERMARCHES_HYPERMARCHES",
        "NB_RESTAURANTS_RESTAURATION_RAPIDE_BPE",
        "EQUIPEMENTS_POUR_10000_HAB"
    ] if c in profil_j5.columns
]
display(profil_j5.loc[profil_j5["CODGEO"].isin(communes_test), colonnes_affichage])
print("Contrôle de Paris, Saint-Denis et des communes tests réussi.")

## 11. Contrôles qualité et export

In [ ]:
assert profil_j5["CODGEO"].is_unique
assert profil_j5["CODGEO"].str.fullmatch(r"\d{5}").all()
assert len(profil_j5) == len(profil_j4)

for col in colonnes_equipements:
    assert (profil_j5[col] >= 0).all(), f"Valeur négative dans {col}."

colonnes_j4_absentes = set(profil_j4.columns) - set(profil_j5.columns)
assert not colonnes_j4_absentes, f"Colonnes du J4 perdues : {sorted(colonnes_j4_absentes)}"

print("Colonnes conservées depuis le J4 :", len(profil_j4.columns))
print("Nombre total de colonnes du J5 :", len(profil_j5.columns))

profil_j5.to_csv(FICHIER_SORTIE, index=False, encoding="utf-8-sig")
print("Fichier créé :", FICHIER_SORTIE)
print("Nombre de communes :", len(profil_j5))
print("J5 terminé ✅")

## Résultat du J5

Fichier produit :

`data/processed/insee/profil_geomarketing_idf_j5.csv`

Il conserve les données des J2 à J4 et ajoute les équipements et services de la BPE 2025.

Dans `docs/sources.csv`, ajoute :

```csv
DS_BPE_CSV_FR.zip;INSEE - Base permanente des équipements 2025;2025;2026;2026-08-24;https://www.insee.fr/fr/statistiques/8217527?sommaire=8217537
```

Enregistre le notebook dans GitHub sous :

`notebooks/05_equipements_services_idf.ipynb`

Résumé du commit : `Ajout des équipements et services BPE 2025`